# Consumo de energía en edificios municipales. Datos mensuales

La información existente en este conjunto de datos proviene de las bases de datos del servicio de seguimiento energético avanzado realizado en el ámbito del Acuerdo Marco de Servicios Energéticos en Instalaciones del Ayuntamiento de Madrid y sus Organismos Autónomos.

En este contexto, y sin perjuicio del seguimiento y control de la facturación energética en la totalidad de edificios municipales, se instalan sensores de consumo energético en algunos de ellos -los que registran mayor consumo, en los que este presenta mayor variabilidad, los que están incluidos en programas específicos de certificación energética o aquellos que cuyas características técnicas u operativas así lo aconsejan- para su supervisión continua y actuación inmediata en caso de incidencia.

### Importar pandas y leer el archivo csv

In [ ]:
import pandas as pd

In [ ]:
df_rawdata = pd.read_csv("https://datos.madrid.es/egobfiles/MANUAL/300430/consumo_energia_edificios.csv", sep=';', header='infer', decimal=',')

In [ ]:
df_rawdata.head()

In [ ]:
df_rawdata.info()

In [ ]:
df_rawdata.describe(include ='all').T

### 1: Se realiza un filtrado de la base para trabajar y conocer el consumo energético electrico anual por distrito

In [ ]:
df_consumo = df_rawdata[['AÑO','DISTRITO','CLASE','GRUPO','UNIDADES','CONSUMO']].sort_values(by='AÑO')
df_filtrado = df_consumo[df_consumo['UNIDADES'] == 'kWh'].copy()
df_filtrado

### 2 y 4: Agregar una columna al agrupar distrito y año para conocer la normalizacion del consumo anual por distrito

In [ ]:
# Primero, crear una copia explícita para evitar el warning
df_filtrado = df_filtrado.copy()

# Función para limpiar y convertir la columna CONSUMO
def limpiar_y_convertir_consumo(serie):
    serie_limpia = serie.astype(str).str.strip()
    
    # Reemplazar valores no numéricos por NaN
    valores_invalidos = ['NO DATA', 'NO DATA PROC', '(No hay datos)', 
                        'SIN DATOS O INFORMACIÓN INCOMPLETA', 'nan']
    for val in valores_invalidos:
        serie_limpia = serie_limpia.replace(val, pd.NA)
    
    # Reemplazar coma decimal por punto
    serie_limpia = serie_limpia.str.replace(',', '.', regex=False)
    
    # Convertir a numérico
    return pd.to_numeric(serie_limpia, errors='coerce')

# Aplicar la función de limpieza
df_filtrado['CONSUMO'] = limpiar_y_convertir_consumo(df_filtrado['CONSUMO'])

# Verificar la conversión
print(f"Valores convertidos exitosamente: {df_filtrado['CONSUMO'].notna().sum()}")
print(f"Total de registros: {len(df_filtrado)}")
print(f"Valores NaN: {df_filtrado['CONSUMO'].isna().sum()}")

# Los consumos negativos son errores de lectura del contador: se descartan antes de normalizar
negativos = (df_filtrado['CONSUMO'] < 0).sum()
df_filtrado = df_filtrado[~(df_filtrado['CONSUMO'] < 0)]
print(f"Lecturas negativas descartadas: {negativos}")

# Normalización (z-score) del consumo dentro de cada distrito y año
df_filtrado['CONSUMO_NORM'] = df_filtrado.groupby(['DISTRITO', 'AÑO'])['CONSUMO'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# Mostrar resultado
df_filtrado[['DISTRITO', 'AÑO', 'CONSUMO', 'CONSUMO_NORM']]

### 3: Se agrega sort value para encontrar los distritos con los edificios más eficientes (menor consumo normalizado)

In [ ]:
df_eficientes = df_filtrado.sort_values(by='CONSUMO_NORM')
df_eficientes

### 5: Identificar nulos y descartar las lecturas sin consumo

In [ ]:
df_eficientes.isnull().values.any()

In [ ]:
df_eficientes.isnull().sum()

In [ ]:
# Solo se descartan las lecturas sin consumo; GRUPO no se usa en el análisis y está vacío en la mayoría de filas
df_limpio = df_eficientes.dropna(subset=['CONSUMO', 'CONSUMO_NORM'])
print(f"Lecturas válidas: {len(df_limpio)} de {len(df_eficientes)}")

In [ ]:
df_limpio.head()

In [ ]:
df_limpio[['CONSUMO', 'CONSUMO_NORM']].isnull().values.any()

In [ ]:
df_limpio2 = df_limpio.copy()

### 6: Utilizacion de la funcion cut() para categorízar el consumo normalizado en percentiles

In [ ]:
# Definir rangos basados en percentiles del consumo normalizado
df_percentiles = df_limpio2['CONSUMO_NORM'].quantile([0, 0.2, 0.4, 0.6, 0.8, 1.0])
print("Percentiles para definir rangos:")
print(df_percentiles)

# Crear rangos personalizados
rangos_eficiencia = [
    df_percentiles[0.0],    # Mínimo
    df_percentiles[0.2],    # 20%
    df_percentiles[0.4],    # 40%
    df_percentiles[0.6],    # 60%
    df_percentiles[0.8],    # 80%
    df_percentiles[1.0]     # Máximo
]

# Etiquetas para las categorías
etiquetas_eficiencia = [
    'Muy Eficiente',
    'Eficiente', 
    'Moderado',
    'Poco Eficiente',
    'Ineficiente'
]

# Aplicar pd.cut()
df_limpio2['CATEGORIA_EFICIENCIA'] = pd.cut(
    df_limpio2['CONSUMO_NORM'], 
    bins=rangos_eficiencia, 
    labels=etiquetas_eficiencia,
    include_lowest=True
)

In [ ]:
df_limpio2

### 7: Grafico de consumo por distrito

In [ ]:
consumo_por_distrito = df_limpio2.groupby('DISTRITO')['CONSUMO'].mean().sort_values(ascending=False)
ax = consumo_por_distrito.plot(kind='bar', figsize=(12, 5), title='Consumo eléctrico medio por lectura y distrito (kWh)')
ax.set_xlabel('Distrito')
ax.set_ylabel('kWh')

### 8: Guardado del DataFrame de consumo de energia electrica de Madrid por distrito

In [ ]:
df_limpio2.to_csv('consumo_energia_madrid_procesado.csv',index=False,sep=';',decimal=',')

### Limitaciones y siguientes pasos
- El z-score se calcula sobre **lecturas** (una fila por sensor y mes), no por edificio. Para comparar edificios convendría agregar primero el consumo por edificio y año, y normalizarlo por superficie o por uso.
- `pd.cut` usa los percentiles del propio conjunto, así que cada categoría agrupa aproximadamente el 20 % de las lecturas.
- Las lecturas de texto ("SIN DATOS…", "NO DATA") se tratan como nulos y se descartan.